# Weight initialization (Xavier)

_Deep Learning_

**Start the weights at small, just-right random values. Zeros or huge values break training.**

Before training, every weight needs a starting value. The choice matters a lot.

     If all weights start at zero, every neuron computes the same thing and they never split apart. Learning is stuck.

---

This notebook is a practice scaffold for the **Weight initialization (Xavier)** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np, matplotlib.pyplot as plt

## First, look at the data

Before training on it, see what each example actually contains. These are **images, not table columns** — each sample is an 8x8 grid of pixel intensities (0–16).

In [ ]:
from sklearn.datasets import load_digits

digits = load_digits()
print("image array:", digits.images.shape, " labels:", digits.target.shape)
samples = list(zip(digits.images, digits.target))
fig, axes = plt.subplots(1, 5, figsize=(8, 2))
for ax, (image, label) in zip(axes, samples):
    ax.imshow(image, cmap="gray")
    ax.set_title(str(label))
    ax.axis("off")
plt.show()

## Reference implementation — PyTorch

In [ ]:
import torch
import torch.nn as nn

net = nn.Sequential(
    nn.Linear(100, 100), nn.ReLU(),
    nn.Linear(100, 100), nn.ReLU(),
    nn.Linear(100, 10),
)

def init_weights(m):
    if isinstance(m, nn.Linear):
        nn.init.kaiming_normal_(m.weight, nonlinearity="relu")  # He init for ReLU
        nn.init.zeros_(m.bias)                                  # biases start at 0

net.apply(init_weights)               # walk every layer and init it

x = torch.randn(32, 100)
out = net(x)
print("output std:", round(out.std().item(), 3))   # stays sane, not 0 or huge

## Visualize the data & results

_Pushing a real batch of digit images through 8 layers, does the activation size stay stable or explode?_

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits

digits = load_digits()
batch = digits.data[:512] / 16.0
batch = batch - batch.mean(0)           # real centered digit batch
rng = np.random.default_rng(3)
nh = 64

def propagate(scale, depth=8):
    x = np.maximum(0, batch @ (rng.standard_normal((64, nh)) * np.sqrt(2 / 64)))
    stds = [x.std()]
    for _ in range(depth):
        x = np.maximum(0, x @ (rng.standard_normal((nh, nh)) * scale))
        stds.append(x.std())
    return stds

he = propagate(np.sqrt(2 / nh))          # He init
big = propagate(np.sqrt(2 / nh) * 2.5)   # too large

plt.plot(he, color="#7ee787", label="He init (good)")
plt.plot(big, color="#ff7b72", label="too-large init (bad)")
plt.title("Activation std across layers on real digit batch (He vs too-large init)")
plt.xlabel("layer"); plt.ylabel("activation std (log scale)")
plt.yscale("log")
plt.legend()
plt.show()